<a href="https://colab.research.google.com/github/AktanM11/AI-OI/blob/main/DAY1_WEEK3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install fastembed

In [ ]:
pip install qdrant-client

In [ ]:
!pip install -U FlagEmbedding

In [4]:
import qdrant_client
from qdrant_client import QdrantClient, models
from qdrant_client.models import VectorParams, PointStruct, Distance, SparseVectorParams, Prefetch
from FlagEmbedding import FlagAutoModel

In [5]:
from fastembed.sparse import SparseTextEmbedding

In [ ]:
dense_model = FlagAutoModel.from_finetuned('BAAI/bge-m3',
                                      use_fp16=True)

In [ ]:
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")

In [8]:
from google.colab import userdata
client = QdrantClient(url=userdata.get('QDRANT_URL'), api_key=userdata.get('QDRANT_API_KEY'))

In [9]:
file_path = "inventory.jsonl"
collection_name = "BGE-M3+BM25"

In [10]:
collections_response = client.get_collections()
collection_names = [col.name for col in collections_response.collections]
print("Existing collections:", collection_names)

Existing collections: ['BGE-M3+BM25', 'inventory_collection', 'items', 'my_books', 'my_knowledge_base', 'policies_collection', 'second_policies_collection', 'test']


In [11]:
collection_info = client.get_collection('BGE-M3+BM25')

In [12]:
print("Dense Vectors Key Name:", list(collection_info.config.params.vectors.keys()) if isinstance(collection_info.config.params.vectors, dict) else "Default (No Name / Empty String)")
print("Sparse Vectors Key Name:", list(collection_info.config.params.sparse_vectors.keys()))

Dense Vectors Key Name: ['text-dense']
Sparse Vectors Key Name: ['text-sparse']


In [14]:
query_text = "зарядка на айфон"

query_dense = dense_model.encode(query_text)['dense_vecs']

query_sparse_output = list(sparse_model.embed([query_text]))[0]

query_sparse = models.SparseVector(
    indices=query_sparse_output.indices.tolist(),
    values=query_sparse_output.values.tolist()
)

# Perform Hybrid Search via RRF
search_result = client.query_points(
    collection_name=collection_name,
    prefetch=[
        # Prefetch 1: Dense Search
        models.Prefetch(
            query=query_dense,
            using="text-dense", # default dense vector
            limit=20
        ),
        # Sparse Search
        models.Prefetch(
            query=query_sparse,
            using="text-sparse",
            limit=20
        )
    ],
    # Merge results using Reciprocal Rank Fusion
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=10
)

for point in search_result.points:
    print(f"ID: {point.id}, Score: {point.score}, Payload: {point.payload}")

ID: 50, Score: 0.5, Payload: {'product_name': 'Apple iPhone 14 128 GB Starlight', 'brand': 'Apple', 'store_name': 'O!Store Вефа', 'store_city': 'Бишкек', 'in_stock': True, 'category': None}
ID: 49, Score: 0.33333334, Payload: {'product_name': 'Apple iPhone 14 128 GB Starlight', 'brand': 'Apple', 'store_name': 'O!Store Джалал Абад Дом Быта', 'store_city': 'Джалал-Абад', 'in_stock': True, 'category': None}
ID: 47, Score: 0.25, Payload: {'product_name': 'Apple iPhone 14 128 GB Starlight', 'brand': 'Apple', 'store_name': 'O!Store ЦУМ O!Store', 'store_city': 'Бишкек', 'in_stock': True, 'category': None}
ID: 48, Score: 0.2, Payload: {'product_name': 'Apple iPhone 14 128 GB Starlight', 'brand': 'Apple', 'store_name': 'O!Store Дордой Плаза ОПиО', 'store_city': 'Бишкек', 'in_stock': True, 'category': None}
ID: 46, Score: 0.16666667, Payload: {'product_name': 'Apple iPhone 14 128 GB Starlight', 'brand': 'Apple', 'store_name': 'O!Store Азия Молл', 'store_city': 'Бишкек', 'in_stock': True, 'catego

In [15]:
query_text = "зарядка на айфон"

query_dense = dense_model.encode(query_text)['dense_vecs']

query_sparse_output = list(sparse_model.embed([query_text]))[0]

query_sparse = models.SparseVector(
    indices=query_sparse_output.indices.tolist(),
    values=query_sparse_output.values.tolist()
)

# Perform Hybrid Search via RRF
search_result = client.query_points(
    collection_name=collection_name,
    prefetch=[
        # Prefetch 1: Dense Search
        models.Prefetch(
            query=query_dense,
            using="text-dense", # default dense vector
            limit=20
        ),
        # Sparse Search
        models.Prefetch(
            query=query_sparse,
            using="text-sparse",
            limit=20
        )
    ],
    # Merge results using Distribution
    query=models.FusionQuery(fusion=models.Fusion.DBSF),
    limit=10
)

for point in search_result.points:
    print(f"ID: {point.id}, Score: {point.score}, Payload: {point.payload}")

ID: 49, Score: 0.69957143, Payload: {'product_name': 'Apple iPhone 14 128 GB Starlight', 'brand': 'Apple', 'store_name': 'O!Store Джалал Абад Дом Быта', 'store_city': 'Джалал-Абад', 'in_stock': True, 'category': None}
ID: 45, Score: 0.69957143, Payload: {'product_name': 'Apple iPhone 14 128 GB Starlight', 'brand': 'Apple', 'store_name': 'O!Store Ош Ареопаг', 'store_city': 'Ош', 'in_stock': True, 'category': None}
ID: 48, Score: 0.69957143, Payload: {'product_name': 'Apple iPhone 14 128 GB Starlight', 'brand': 'Apple', 'store_name': 'O!Store Дордой Плаза ОПиО', 'store_city': 'Бишкек', 'in_stock': True, 'category': None}
ID: 50, Score: 0.69957143, Payload: {'product_name': 'Apple iPhone 14 128 GB Starlight', 'brand': 'Apple', 'store_name': 'O!Store Вефа', 'store_city': 'Бишкек', 'in_stock': True, 'category': None}
ID: 47, Score: 0.69957143, Payload: {'product_name': 'Apple iPhone 14 128 GB Starlight', 'brand': 'Apple', 'store_name': 'O!Store ЦУМ O!Store', 'store_city': 'Бишкек', 'in_stoc

In [ ]:
query_text = "BA52A"

query_dense = dense_model.encode(query_text)['dense_vecs']

query_sparse_output = list(sparse_model.embed([query_text]))[0]

query_sparse = models.SparseVector(
    indices=query_sparse_output.indices.tolist(),
    values=query_sparse_output.values.tolist()
)

# Perform Hybrid Search via RRF
search_result = client.query_points(
    collection_name=collection_name,
    prefetch=[
        # Prefetch 1: Dense Search
        models.Prefetch(
            query=query_dense,
            using="text-dense", # default dense vector
            limit=20
        ),
        # Sparse Search
        models.Prefetch(
            query=query_sparse,
            using="text-sparse",
            limit=20
        )
    ],
    # Merge results using Reciprocal Rank Fusion
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=10
)

for point in search_result.points:
    print(f"ID: {point.id}, Score: {point.score}, Payload: {point.payload}")

ID: 7804, Score: 0.5, Payload: {'product_name': 'ЗУ Borofone BA52A Micro', 'brand': 'Unknown', 'store_name': 'O!Store Ноокат 2', 'store_city': 'Ноокат', 'in_stock': True, 'category': None}
ID: 7815, Score: 0.33333334, Payload: {'product_name': 'ЗУ Borofone BA52A Micro', 'brand': 'Unknown', 'store_name': 'O!Store Табылга', 'store_city': 'Бишкек', 'in_stock': True, 'category': None}
ID: 7812, Score: 0.25, Payload: {'product_name': 'ЗУ Borofone BA52A Micro', 'brand': 'Unknown', 'store_name': 'O!Store Балыкчы Глобус', 'store_city': 'Балыкчы', 'in_stock': True, 'category': None}
ID: 7805, Score: 0.2, Payload: {'product_name': 'ЗУ Borofone BA52A Micro', 'brand': 'Unknown', 'store_name': 'O!Store Берекет Гранд', 'store_city': 'Бишкек', 'in_stock': True, 'category': None}
ID: 7810, Score: 0.16666667, Payload: {'product_name': 'ЗУ Borofone BA52A Micro', 'brand': 'Unknown', 'store_name': 'O!Store Кызыл Кия Армада', 'store_city': 'Кызыл-Кия', 'in_stock': True, 'category': None}
ID: 7803, Score: 0

In [ ]:
query_text = "16 Pro Max"

query_dense = dense_model.encode(query_text)['dense_vecs']

query_sparse_output = list(sparse_model.embed([query_text]))[0]

query_sparse = models.SparseVector(
    indices=query_sparse_output.indices.tolist(),
    values=query_sparse_output.values.tolist()
)

# Perform Hybrid Search via RRF
search_result = client.query_points(
    collection_name=collection_name,
    prefetch=[
        # Prefetch 1: Dense Search
        models.Prefetch(
            query=query_dense,
            using="text-dense", # default dense vector
            limit=20
        ),
        # Sparse Search
        models.Prefetch(
            query=query_sparse,
            using="text-sparse",
            limit=20
        )
    ],
    # Merge results using Reciprocal Rank Fusion
    # score = 0.0
    # for q in queries:
    # if d in result(q):
    #    score += 1.0 / ( k + rank( result(q), d ) )
    # return score
    # k is a ranking constant
    # q is a query in the set of queries
    # d is a document in the result set of q
    # result(q) is the result set of q
    # rank( result(q), d ) is d's rank within the result(q) starting from 1
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=10
)

for point in search_result.points:
    print(f"ID: {point.id}, Score: {point.score}, Payload: {point.payload}")

ID: 104, Score: 0.8337896, Payload: {'product_name': 'Apple iPhone 16 Pro Max 256 GB Black Titanium', 'brand': 'Apple', 'store_name': 'O!Store Томми Молл', 'store_city': 'Бишкек', 'in_stock': True, 'category': None}
ID: 0, Score: 0.82489014, Payload: {'product_name': 'Apple iPhone 16 Pro Max 256 GB Natural Titanium', 'brand': 'Apple', 'store_name': 'O!Store Головной', 'store_city': 'Бишкек', 'in_stock': True, 'category': None}
ID: 53, Score: 0.8173046, Payload: {'product_name': 'Apple iPhone 16 Pro Max 256 GB White Titanium', 'brand': 'Apple', 'store_name': 'O!Store Талас', 'store_city': 'Талас', 'in_stock': True, 'category': None}
ID: 89, Score: 0.7058035, Payload: {'product_name': 'Apple iPhone 16 Pro Max 512 GB Desert Titanium', 'brand': 'Apple', 'store_name': 'O!Store ЦУМ O!Store', 'store_city': 'Бишкек', 'in_stock': True, 'category': None}
ID: 103, Score: 0.69272655, Payload: {'product_name': 'Apple iPhone 16 Pro Max 256 GB Desert Titanium', 'brand': 'Apple', 'store_name': 'O!Stor